DecisionTreeClassifier 用户流失预测（分类）

In [32]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')


In [88]:
# 1、读取数据
df = pd.read_csv("../../../data/ml/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [89]:
# 2、数据预处理
# 字符串的列
# select_dtypes 选择符合条件的列， .columns取出列标签
cat_cols = df.select_dtypes(include='object').columns.tolist()
# 数字数据的列
num_cols = df.select_dtypes(exclude='object').columns.tolist()
print("字符串的列: ", cat_cols)
print("数字数据的列: ", num_cols)


字符串的列:  ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']
数字数据的列:  ['SeniorCitizen', 'tenure', 'MonthlyCharges']


In [90]:
# 查看字符串数据列的每一列的枚举值是多少个
print(df[cat_cols].nunique())
df[cat_cols].nunique()

customerID          7043
gender                 2
Partner                2
Dependents             2
PhoneService           2
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
DeviceProtection       3
TechSupport            3
StreamingTV            3
StreamingMovies        3
Contract               3
PaperlessBilling       2
PaymentMethod          4
TotalCharges        6531
Churn                  2
dtype: int64


customerID          7043
gender                 2
Partner                2
Dependents             2
PhoneService           2
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
DeviceProtection       3
TechSupport            3
StreamingTV            3
StreamingMovies        3
Contract               3
PaperlessBilling       2
PaymentMethod          4
TotalCharges        6531
Churn                  2
dtype: int64

In [91]:
# customerID 是每个客户的id，每个客户都不一样，不具有训练意义，直接删除
df = df.drop(columns=["customerID"])

In [92]:
# TotalCharges 是字符串，这里转化为 数字类型
df["TotalCharges"] = (df["TotalCharges"].str.strip()).replace("" , 0).astype("float")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

In [100]:
# 2、数据预处理 之 使用 OneHotEncoder 对类别数据进行独热编码

# 因为 输入训练的数据必须是 数字 ，字符串类别必须先转化为 数字 数据

# 先把特征和目标分离出来，因为目标target不需要进行独热编码
X = df.drop(columns=['Churn'])
y = df['Churn']

# 3、数据分割
# train:test = 8:2
X_train_src, X_test_src, y_train_src, y_test_src = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train_src.shape, y_train_src.shape)
print(X_test_src.shape, y_test_src.shape)

# 抽取出数字类别和类型类别
numerical_cols = X_train_src.select_dtypes(include='number').columns.tolist()
categorical_cols = X_train_src.select_dtypes(exclude='number').columns.tolist()
print("数字数据的列: ", numerical_cols)
print("类别的列: ", categorical_cols)

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# 先使用训练数据对 encoder 进行编码拟合
X_train_src_encoded = pd.DataFrame(
    encoder.fit_transform(X_train_src[categorical_cols]),
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_train_src.index
)
# 将 独热编码的数据 和 不需要独热编码的原始数据 进行拼接，得到最终的特征数据
X_train = pd.concat([X_train_src_encoded, X_train_src.drop(columns=categorical_cols)], axis=1)
print("X_train: ", X_train.columns, len(X_train.columns))

# 再使用 拟合好的encoder 对 X_test 进行转化
X_test_src_encoded = pd.DataFrame(
    encoder.transform(X_test_src[categorical_cols]),
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_test_src.index
)
X_test = pd.concat([X_test_src_encoded, X_test_src.drop(columns=categorical_cols)], axis=1)
print("X_test: ", X_test.columns, len(X_test.columns))

# 统一列名类型
# TypeError: Feature names are only supported if all input features have string names, but your input has ['int', 'str'] as feature name / column name types
X_train.columns = X_train.columns.astype(str)

(5634, 19) (5634,)
(1409, 19) (1409,)
数字数据的列:  ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
类别的列:  ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
X_train:  Index(['gender_Female', 'gender_Male', 'Partner_No', 'Partner_Yes',
       'Dependents_No', 'Dependents_Yes', 'PhoneService_No',
       'PhoneService_Yes', 'MultipleLines_No',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_DSL', 'InternetService_Fiber optic',
       'InternetService_No', 'OnlineSecurity_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No', 'OnlineBackup_No internet service',
       'OnlineBackup_Yes', 'DeviceProtection_No',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No', 'TechSuppo

In [114]:
# 2、数据预处理 之 对目标标签 target 的处理
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
print(y_train_src.shape, y_test_src.shape)
y_train = label_encoder.fit_transform(y_train_src)
y_test = label_encoder.transform(y_test_src)
# tmp1 = np.column_stack((y_train_src, y_train))
# tmp1
tmp2 = np.column_stack((y_test_src, y_test))
tmp2

(5634,) (1409,)


array([['No', 0],
       ['No', 0],
       ['No', 0],
       ...,
       ['No', 0],
       ['No', 0],
       ['No', 0]], shape=(1409, 2), dtype=object)

In [125]:
# 4、定义模型
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", DecisionTreeClassifier(max_features=10, random_state=23))
])
model.fit(X_train, y_train)

print(model.named_steps["classifier"].get_depth())
print(model.named_steps["classifier"].get_n_leaves())

24
1543


In [122]:
# 5、模型评估
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.7317246273953159
              precision    recall  f1-score   support

           0       0.82      0.82      0.82      1035
           1       0.49      0.50      0.50       374

    accuracy                           0.73      1409
   macro avg       0.66      0.66      0.66      1409
weighted avg       0.73      0.73      0.73      1409

[[844 191]
 [187 187]]


In [123]:
# 8. 预测新客户是否会流失
X_validation = X_test[0:50]
y_validation = y_test[0:50]

y_validation_pred = model.predict(X_validation)
res = np.column_stack((y_validation, y_validation_pred, y_validation == y_validation_pred))
print(res)
print(f"accuracy_score: {accuracy_score(y_validation, y_validation_pred)}")
print(f"accuracy_score2: {res[:, -1].sum() / len(y_validation):.2f}")


[[0 0 1]
 [0 1 0]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [1 1 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [1 1 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [1 0 0]
 [0 1 0]
 [0 0 1]
 [0 0 1]
 [0 1 0]
 [0 0 1]
 [1 0 0]
 [1 1 1]
 [0 0 1]
 [1 1 1]
 [0 1 0]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [1 1 1]
 [0 0 1]
 [1 1 1]
 [0 0 1]
 [1 1 1]
 [1 0 0]
 [0 0 1]
 [0 1 0]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [1 1 1]
 [0 0 1]
 [0 0 1]
 [0 0 1]
 [1 0 0]]
accuracy_score: 0.82
accuracy_score2: 0.82
